# Churn Prediction - projeto profissional de Machine Learning

Objetivo: prever quais clientes tem maior probabilidade de cancelar.
O ponto mais importante de engenharia e evitar data leakage: todo
pre-processamento fica dentro de pipelines ajustados apenas no treino.

In [2]:
import sys
sys.path.insert(0, r'C:\\Users\\andre\\Downloads\\ModelCraft')

from src.config import ProjectConfig
from src.data import load_or_create_dataset, summarize_data
from src.eda import run_eda
from src.evaluation import evaluate_on_test, save_business_report
from src.interpretability import create_interpretability_report
from src.leakage_checks import validate_no_obvious_leakage
from src.models import run_model_selection, tune_best_model
from src.preprocessing import build_preprocessor, make_train_valid_test_split
from src.utils import ensure_directories

In [3]:
config = ProjectConfig()
ensure_directories(config)
df = load_or_create_dataset(config)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,SYN-00000,Male,No,Yes,No,47,Yes,Yes,Fiber optic,Yes,...,No,No,No,No,One year,No,Electronic check,40.96,2044.83,No
1,SYN-00001,Female,No,Yes,No,52,Yes,No phone service,Fiber optic,No,...,Yes,Yes,No,No internet service,Month-to-month,Yes,Mailed check,83.51,4295.33,No
2,SYN-00002,Female,No,No,No,20,Yes,No,DSL,Yes,...,No internet service,No,No internet service,No,Two year,Yes,Bank transfer (automatic),52.99,988.95,No
3,SYN-00003,Male,No,Yes,Yes,26,Yes,No,No,Yes,...,Yes,Yes,No,No internet service,One year,No,Mailed check,23.83,727.08,No
4,SYN-00004,Male,No,Yes,No,69,Yes,Yes,Fiber optic,No internet service,...,No,Yes,No,No internet service,Month-to-month,No,Credit card (automatic),61.95,3980.21,No


In [4]:
summarize_data(df, config)
run_eda(df, config)
df.shape, df.dtypes, df.isna().sum().sort_values(ascending=False).head()

((4000, 21),
 customerID           object
 gender               object
 SeniorCitizen        object
 Partner              object
 Dependents           object
 tenure                int64
 PhoneService         object
 MultipleLines        object
 InternetService      object
 OnlineSecurity       object
 OnlineBackup         object
 DeviceProtection     object
 TechSupport          object
 StreamingTV          object
 StreamingMovies      object
 Contract             object
 PaperlessBilling     object
 PaymentMethod        object
 MonthlyCharges      float64
 TotalCharges        float64
 Churn                object
 dtype: object,
 TotalCharges     40
 gender            0
 SeniorCitizen     0
 Partner           0
 customerID        0
 dtype: int64)

In [5]:
X_train, X_valid, X_test, y_train, y_valid, y_test = make_train_valid_test_split(
    df, config.target_column, config.random_state
)
validate_no_obvious_leakage(X_train, X_valid, X_test, config.target_column)

[]

In [6]:
preprocessor = build_preprocessor(X_train)
selection = run_model_selection(X_train, y_train, X_valid, y_valid, preprocessor, config)
selection.leaderboard

c:\Users\andre\Downloads\ModelCraft\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\andre\Downloads\ModelCraft\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\andre\Downloads\ModelCraft\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\andre\Downloads\ModelCraft\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\andre\Downloads\ModelCraft\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier wa

,model,cv_roc_auc_mean,cv_roc_auc_std,train_roc_auc_mean,valid_roc_auc,valid_average_precision,valid_recall,valid_f1,overfit_gap_auc
1,logistic_regression,0.745552,0.014715,0.761582,0.750818,0.710296,0.642077,0.638587,0.016029
2,random_forest,0.720148,0.013013,1.000000,0.735338,0.694588,0.581967,0.614719,0.279852
6,catboost,0.725966,0.013718,0.924477,0.733613,0.695295,0.628415,0.623306,0.198511
4,xgboost,0.715460,0.016695,0.933871,0.726782,0.674245,0.581967,0.614719,0.218411
3,hist_gradient_boosting,0.688646,0.016451,0.999210,0.705472,0.657937,0.598361,0.606648,0.310564
5,lightgbm,0.692626,0.015950,0.999016,0.699844,0.647990,0.581967,0.597475,0.306391
0,dummy_baseline,0.500000,0.000000,0.500000,0.500000,0.457500,0.000000,0.000000,0.000000


In [7]:
tuned = tune_best_model(selection.best_model_name, X_train, y_train, preprocessor, config)
tuned.best_params_

{'model__C': 0.1}

In [8]:
metrics = evaluate_on_test(tuned.best_estimator_, X_test, y_test, config)
metrics

{'accuracy': 0.685,
 'precision': 0.6565934065934066,
 'recall': 0.6530054644808743,
 'f1': 0.6547945205479452,
 'roc_auc': 0.7364332300873814,
 'average_precision': 0.6924498047746431}

In [9]:
create_interpretability_report(tuned.best_estimator_, X_train, X_test, y_test, config)
save_business_report(selection, tuned, metrics, config)
print('Concluido!')

c:\Users\andre\Downloads\ModelCraft\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Concluido!
